# Estruturação e Modelagem dos Dados Processuais — TRT18 (1º Grau)

Pipeline que transforma os dados coletados do PJe do TRT18 em uma **árvore JSON padronizada**
para consumo pelo chatbot. Segue as etapas do `tarefas.md`:

1. Recebimento da base de processos (planilha)
2. Coleta dos dados (Playwright + CAPTCHA + extração)
3. Análise da estrutura dos dados
4. Construção da árvore JSON (esquema Pydantic)
5. Limpeza e padronização
6. Validação dos dados (Pydantic)
7. Geração do arquivo JSON único
8. Entrega para o chatbot

> **Módulos auxiliares:** `modelos.py` (esquema), `coletor_trt18.py` (raspagem/extração), `normalizador.py` (limpeza + mapeamento + validação), `servico.py` (integração com o chatbot).


> **Nota.** Este notebook documenta a etapa de *pesquisa* do trabalho: a coleta
> em lote da amostra de 100 processos e a análise da estrutura dos dados
> (etapas 1→8 do `tarefas.md`). **Não é o fluxo de produção.**
>
> O fluxo em uso é a consulta sob demanda de um processo por vez, via
> `app.py` (linha de comando) ou `servico.obter_processo()` (chatbot).
>
> Rode-o a partir da pasta `pesquisa/` ou da raiz do projeto.


## Configuração


In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

# Este notebook vive em `pesquisa/`; os módulos e os dados ficam na raiz do
# projeto. Resolvemos a raiz para que ele rode a partir de qualquer um dos dois.
RAIZ = Path.cwd().parent if Path.cwd().name == 'pesquisa' else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from modelos import Processo, SituacaoProcesso, StatusColeta
from normalizador import normalizar, extrair_campos
import coletor_trt18 as coletor

DIR_ENTRADA = RAIZ / 'data' / 'entrada'
DIR_BRUTOS  = RAIZ / 'data' / 'brutos'
DIR_SAIDA   = RAIZ / 'data' / 'saida'
DIR_BRUTOS.mkdir(parents=True, exist_ok=True)
DIR_SAIDA.mkdir(parents=True, exist_ok=True)

PLANILHA = next(DIR_ENTRADA.glob('*.xlsx'))
print('Raiz do projeto:', RAIZ)
print('Planilha de entrada:', PLANILHA.name)

## Etapa 1 — Recebimento da base de processos

Lê a planilha fornecida pela professora (~100 processos) e padroniza a lista de números.


In [ ]:
df_base = pd.read_excel(PLANILHA, usecols=[0, 1], names=['processo', 'status_cnj'], header=0)
df_base = df_base.dropna(subset=['processo'])
df_base['processo'] = df_base['processo'].astype(str).str.strip()
df_base['status_cnj'] = df_base['status_cnj'].astype(str).str.strip()

processos = [
    {'numero': r.processo, 'status_cnj_planilha': r.status_cnj}
    for r in df_base.itertuples(index=False)
]
print(f'{len(processos)} processos na base.')
df_base.head()

## Etapa 2 — Coleta dos dados

A coleta abre um navegador real, resolve o CAPTCHA e captura os dados do processo
(respostas da API do PJe + HTML/texto como fallback). Cada bruto é salvo em
`data/brutos/<numero>.json`, permitindo retomar lotes sem re-raspar.

> ⚠️ **Para coletar os 100 processos de verdade**, rode na sua máquina com
> `COLETAR_AGORA = True` (precisa de navegador + resolver CAPTCHA).

> Se `COLETAR_AGORA = False` e ainda **não houver nada** em `data/brutos/`, o notebook
> entra em **MODO DEMONSTRAÇÃO**: usa só o `exemplos/exemplo_bruto.json` (1 processo)
> e grava num arquivo `*_DEMO.json` — para você não confundir com o resultado real.


In [ ]:
COLETAR_AGORA = False  # << mude para True para raspar os 100 processos da planilha

if COLETAR_AGORA:
    brutos = coletor.coletar_lote(processos, headless=False, pular_existentes=True)
    MODO_DEMO = False
else:
    brutos = [json.loads(p.read_text(encoding='utf-8')) for p in sorted(DIR_BRUTOS.glob('*.json'))]
    MODO_DEMO = len(brutos) == 0
    if MODO_DEMO:
        exemplo = RAIZ / 'exemplos' / 'exemplo_bruto.json'
        brutos = [json.loads(exemplo.read_text(encoding='utf-8'))]

# cobertura em relação à base de 100 processos
nums_base = {''.join(c for c in p['numero'] if c.isdigit()) for p in processos}
nums_coletados = {b.get('numero_normalizado') for b in brutos}
faltando = nums_base - nums_coletados

if MODO_DEMO:
    print('=' * 70)
    print('  AVISO: MODO DEMONSTRACAO -- nenhum processo foi coletado de verdade.')
    print('         Usando apenas exemplos/exemplo_bruto.json (1 processo).')
    print('         Para gerar os 100: defina COLETAR_AGORA = False e rode de novo.')
    print('=' * 70)
else:
    print(f'{len(brutos)} processos brutos carregados.')
    print(f'Cobertura: {len(nums_base) - len(faltando)}/{len(nums_base)} da base '
          f'({len(faltando)} ainda faltando).')

## Etapa 3 — Análise da estrutura dos dados

Antes de fixar o modelo, inspecionamos os campos presentes/ausentes para entender
padrões e inconsistências entre os processos.


In [ ]:
df_analise = pd.DataFrame([extrair_campos(b) for b in brutos])
print('Preenchimento por campo (%):')
display((df_analise.notna().mean() * 100).round(1).sort_values(ascending=False))
df_analise.head()

## Etapas 4, 5 e 6 — Estruturação, limpeza e validação

`normalizar()` faz tudo de uma vez: monta a árvore JSON (Etapa 4), aplica limpeza/
normalização de texto, datas e valores (Etapa 5) e valida via Pydantic (Etapa 6).
Processos que falharem na validação são separados para inspeção.


In [ ]:
processos_validos: list[Processo] = []
falhas: list[dict] = []

for bruto in brutos:
    try:
        processos_validos.append(normalizar(bruto))
    except Exception as exc:
        falhas.append({'numero': bruto.get('numero'), 'erro': str(exc)})

print(f'Validados: {len(processos_validos)} | Falhas: {len(falhas)}')
if falhas:
    display(pd.DataFrame(falhas))

resumo = pd.DataFrame([{
    'numero': p.processo.numero,
    'situacao': p.processo.situacao.value,
    'status_coleta': p.coleta.status_coleta.value,
    'n_partes': len(p.partes_interessadas),
    'n_movimentacoes': len(p.movimentacoes),
    'n_documentos': len(p.documentos),
} for p in processos_validos])
resumo

## Etapa 7 — Geração do arquivo JSON único

Todos os processos validados em um único array. Em **modo demonstração** o arquivo é
gravado como `processos_trt18_DEMO.json`, para nunca sobrescrever o resultado real.


In [ ]:
nome_saida = 'processos_trt18_DEMO.json' if MODO_DEMO else 'processos_trt18.json'
ARQUIVO_SAIDA = DIR_SAIDA / nome_saida

payload = [json.loads(p.model_dump_json()) for p in processos_validos]
ARQUIVO_SAIDA.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'{len(payload)} processo(s) gravado(s) em: {ARQUIVO_SAIDA.name}')
if MODO_DEMO:
    print('AVISO: arquivo de DEMONSTRACAO (1 processo). NAO e a entrega final.')
else:
    print(f'Tamanho: {ARQUIVO_SAIDA.stat().st_size} bytes')

## Etapa 8 — Entrega para o chatbot

O arquivo `data/saida/processos_trt18.json` é o produto final desta etapa. O chatbot
não acessa o PJe diretamente — consome apenas estes dados já organizados e validados
(via `servico.obter_processo`). Abaixo, um exemplo de consulta rápida por número.


In [ ]:
def consultar(numero: str) -> dict | None:
    alvo = ''.join(ch for ch in numero if ch.isdigit())
    for proc in payload:
        if proc['processo']['numero_normalizado'] == alvo:
            return proc
    return None

exemplo = consultar(payload[0]['processo']['numero'])
print(json.dumps(exemplo, ensure_ascii=False, indent=2)[:1200])